<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-03-arvores/Aula%2023%20-%20Protocolo%20de%20%C3%81rvore%20Geradora%20(STP%20-%20RSTP)%20-%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Simulação do Protocolo de Árvore Geradora (STP / RSTP) no SCADA-Core
### Topologia em Anel Redundante da Planta Piloto de Biodiesel (Grupo 5)

**Unidade Operacional:** Rede Ethernet Industrial (Camada 2) dos CLPs e Servidores da Planta de Biodiesel  
**Infraestrutura Modelada:**
1. **`SW-CORE`**: Switch Central da Sala de Controle (SCADA-Core e Historiador);
2. **`SW-ZONA1`**: Switch da Zona de Insumos (Setor 100 — Tanques TK-101, TK-102 e TK-103);
3. **`SW-ZONA2`**: Switch da Zona de Reação e Decantação (Setores 200/300 — Reator CSTR R-200 e Decantador TK-300);
4. **`SW-ZONA3`**: Switch da Zona de Purificação e Expedição (Setor 400 — Coluna COL-401 e Tanque B100 TK-402).

**Objetivos do Estudo:**
- Simular a eleição determinística do **Root Bridge** e cálculo dos custos de caminho conforme **IEEE 802.1w (RSTP)**;
- Prevenir loops de Camada 2 (*Broadcast Storms*) bloqueando logicamente arestas redundantes (*Alternate Ports / Discarding*);
- Simular a quebra física de link no backbone principal do Reator R-200 e demonstrar a reconvergência automática para a árvore geradora reserva.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, Tuple, List, Any

# Configurações estéticas de gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print("✅ Módulos de redes e teoria dos grafos importados com sucesso!")

---
## 1. Modelagem da Rede Industrial como Grafo Valorado $G = (V, E, w)$

Configuramos as prioridades dos switches para garantir que o switch central (`SW-CORE`) vença a eleição como Root Bridge primário (prioridade `4096`) e a Zona 1 assuma como Root Bridge secundário (prioridade `8192`) caso ocorra falha completa na sala de controle.

Os custos dos enlaces seguem a tabela padrão da norma **IEEE 802.1w**:
- **10 Gbps (Backbone Fibra):** Custo = $2.000$;
- **1 Gbps (Anel Fibra):** Custo = $20.000$;
- **100 Mbps (Cobre Backup):** Custo = $200.000$.

In [ ]:
# 1. Definição dos Switches Industriais (Vértices V)
# Bridge ID = (Priority, MAC) -> Menor BID vence a eleição
switches_biodiesel = {
    "SW-CORE":  {"zona": "Sala de Controle SCADA-Core",          "priority": 4096,  "mac": "00:1A:2B:00:00:01"},
    "SW-ZONA1": {"zona": "Zona 1 (Insumos TK-101/102/103)",       "priority": 8192,  "mac": "00:1A:2B:00:00:02"},
    "SW-ZONA2": {"zona": "Zona 2 (Reator R-200 / Decantador)", "priority": 32768, "mac": "00:1A:2B:00:00:03"},
    "SW-ZONA3": {"zona": "Zona 3 (Expedição B100 COL-401)",     "priority": 32768, "mac": "00:1A:2B:00:00:04"},
}

# 2. Definição dos Enlaces Físicos (Arestas E e Pesos w)
enlaces_fisicos = [
    ("SW-CORE", "SW-ZONA2", {"cost": 2000,   "tipo": "10 Gbps (Fibra Tronco Reator R-200)"}),
    ("SW-CORE", "SW-ZONA1", {"cost": 20000,  "tipo": "1 Gbps (Fibra Anel Insumos)"}),
    ("SW-CORE", "SW-ZONA3", {"cost": 20000,  "tipo": "1 Gbps (Fibra Anel Expedição)"}),
    ("SW-ZONA1", "SW-ZONA2", {"cost": 20000,  "tipo": "1 Gbps (Fibra Interzonal)"}),
    ("SW-ZONA2", "SW-ZONA3", {"cost": 20000,  "tipo": "1 Gbps (Fibra Interzonal)"}),
    ("SW-ZONA1", "SW-ZONA3", {"cost": 200000, "tipo": "100 Mbps (Cabo Cobre Redundante)"})
]

G_rede = nx.Graph()
for sw, attrs in switches_biodiesel.items():
    G_rede.add_node(sw, **attrs)
G_rede.add_edges_from(enlaces_fisicos)

num_v = G_rede.number_of_nodes()
num_e = G_rede.number_of_edges()
ciclos = num_e - num_v + 1  # Número ciclomático do grafo

print(f"Grafo da Rede SCADA criado: |V| = {num_v} switches e |E| = {num_e} enlaces físicos.")
print(f"Ciclos independentes detectados (anéis físicos redundantes): {ciclos}")
assert ciclos == 3, "A topologia com 4 nós e 6 arestas deve possuir exatamente 3 ciclos fundamentais."

---
## 2. Implementação do Motor de Convergência do RSTP (IEEE 802.1w)

A função `executar_rstp()` reproduz o comportamento determinístico do protocolo:
1. Identifica o **Root Bridge** com o menor Bridge ID ($\text{BID} = (\text{Priority}, \text{MAC})$);
2. Calcula o **Root Path Cost** de cada switch até a raiz via caminho de menor custo acumulado;
3. Define a **Root Port (RP)** de cada nó não-raiz em direção ao Root Bridge;
4. Classifica as demais portas em **Designated Port (DP)** (estado *Forwarding*) ou **Alternate Port (AP)** (estado *Discarding / Bloqueada*).

In [ ]:
def executar_rstp(grafo: nx.Graph) -> Tuple[str, Dict[str, int], List[Tuple], List[Tuple], pd.DataFrame]:
    """Simula a convergência da Árvore Geradora RSTP sobre a topologia de switches."""
    if not nx.is_connected(grafo):
        raise ValueError("Falha crítica: Grafo desconexo! A rede foi particionada.")

    # 1. Eleição do Root Bridge
    root_bridge = min(
        grafo.nodes,
        key=lambda n: (grafo.nodes[n]["priority"], grafo.nodes[n]["mac"])
    )

    # 2. Cálculo do Root Path Cost (Dijkstra centrado no Root Bridge)
    root_path_cost = {n: float("inf") for n in grafo.nodes}
    root_port_parent = {n: None for n in grafo.nodes}
    root_path_cost[root_bridge] = 0

    visitados = set()
    while len(visitados) < len(grafo.nodes):
        u = min(
            (n for n in grafo.nodes if n not in visitados),
            key=lambda n: (root_path_cost[n], grafo.nodes[n]["priority"], grafo.nodes[n]["mac"])
        )
        visitados.add(u)

        for v in grafo.neighbors(u):
            if v in visitados:
                continue
            custo_aresta = grafo.edges[u, v]["cost"]
            novo_custo = root_path_cost[u] + custo_aresta

            atual_pai = root_port_parent[v]
            bid_u = (grafo.nodes[u]["priority"], grafo.nodes[u]["mac"])
            bid_pai = (grafo.nodes[atual_pai]["priority"], grafo.nodes[atual_pai]["mac"]) if atual_pai else (float("inf"), "")

            if (novo_custo < root_path_cost[v]) or (novo_custo == root_path_cost[v] and bid_u < bid_pai):
                root_path_cost[v] = novo_custo
                root_port_parent[v] = u

    # 3. Classificação de Papéis e Estados de Portas em Cada Enlace
    arestas_ativas = []
    arestas_bloqueadas = []
    registros_tabela = []

    for u, v, data in grafo.edges(data=True):
        if root_port_parent[v] == u:
            papel_u, est_u = "Designated Port (DP)", "Forwarding"
            papel_v, est_v = "Root Port (RP)", "Forwarding"
            arestas_ativas.append((u, v))
        elif root_port_parent[u] == v:
            papel_u, est_u = "Root Port (RP)", "Forwarding"
            papel_v, est_v = "Designated Port (DP)", "Forwarding"
            arestas_ativas.append((u, v))
        else:
            crit_u = (root_path_cost[u], grafo.nodes[u]["priority"], grafo.nodes[u]["mac"])
            crit_v = (root_path_cost[v], grafo.nodes[v]["priority"], grafo.nodes[v]["mac"])

            if crit_u < crit_v:
                papel_u, est_u = "Designated Port (DP)", "Forwarding"
                papel_v, est_v = "Alternate Port (AP)", "Discarding (Bloqueada)"
            else:
                papel_u, est_u = "Alternate Port (AP)", "Discarding (Bloqueada)"
                papel_v, est_v = "Designated Port (DP)", "Forwarding"
            arestas_bloqueadas.append((u, v))

        status_link = "ATIVO (Na Árvore T)" if (u, v) in arestas_ativas else "BLOQUEADO (Loop Discarding)"
        registros_tabela.append({
            "Enlace Físico": f"{u} <---> {v}",
            "Custo Enlace": data["cost"],
            f"Porta em {u}": f"{papel_u} [{est_u}]",
            f"Porta em {v}": f"{papel_v} [{est_v}]",
            "Status Lógico": status_link
        })

    df_resultado = pd.DataFrame(registros_tabela)
    return root_bridge, root_path_cost, arestas_ativas, arestas_bloqueadas, df_resultado

root, custos, ativas, bloqueadas, df_rstp = executar_rstp(G_rede)

print("=================================================================")
print("RESULTADO DA CONVERGÊNCIA RSTP (OPERAÇÃO NOMINAL)")
print("=================================================================")
print(f"Root Bridge Eleito  : {root} (Prioridade: {G_rede.nodes[root]['priority']})")
print("Custos Acumulados até o Root Bridge:")
for sw, c in custos.items():
    print(f"  • {sw:10s} ({G_rede.nodes[sw]['zona']:35s}): Custo = {c}")

assert root == "SW-CORE", "SW-CORE deve ser eleito Root Bridge devido à menor prioridade (4096)."
assert len(ativas) == len(G_rede.nodes) - 1, "A árvore geradora deve ter exatamente |V|-1 arestas ativas."
print("\nTabela de Estados das Portas dos Switches:")
df_rstp

---
## 3. Visualização Gráfica da Topologia Física vs. Árvore Geradora Lógica

Desenhamos o grafo da rede supervisória destacando:
- **Vértice Dourado:** Root Bridge Central (`SW-CORE`);
- **Arestas Verdes Contínuas:** Enlaces ativos em *Forwarding* pertencentes à Árvore Geradora $T$;
- **Arestas Vermelhas Tracejadas:** Enlaces mantidos em *Discarding* para quebra de ciclos.

In [ ]:
posicoes_switches = {
    "SW-CORE":  (0.0, 2.0),
    "SW-ZONA1": (-2.0, 0.0),
    "SW-ZONA2": (0.0, -1.8),
    "SW-ZONA3": (2.0, 0.0)
}

def plotar_topologia_rstp(grafo: nx.Graph, ativas: List, bloqueadas: List, root: str, titulo: str):
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # Cores dos nós
    cores_nos = ['#f59e0b' if n == root else '#3b82f6' for n in grafo.nodes()]
    nx.draw_networkx_nodes(grafo, posicoes_switches, node_color=cores_nos, node_size=2800, edgecolors='black', linewidths=2.0, ax=ax)
    nx.draw_networkx_labels(grafo, posicoes_switches, font_size=10, font_weight='bold', font_color='white', ax=ax)
    
    # Arestas ativas
    nx.draw_networkx_edges(grafo, posicoes_switches, edgelist=ativas, edge_color='#10b981', width=3.5, label='Enlace Ativo (Forwarding)', ax=ax)
    # Arestas bloqueadas
    nx.draw_networkx_edges(grafo, posicoes_switches, edgelist=bloqueadas, edge_color='#ef4444', width=2.5, style='dashed', label='Bloqueado RSTP (Discarding)', ax=ax)
    
    # Rótulos de pesos das arestas
    rotulos_arestas = {(u, v): f"{data['cost']}\n({data['tipo'].split()[0]})" for u, v, data in grafo.edges(data=True)}
    nx.draw_networkx_edge_labels(grafo, posicoes_switches, edge_labels=rotulos_arestas, font_size=8.5, font_color='#1e293b', ax=ax)
    
    ax.set_title(titulo, fontsize=12, fontweight='bold', pad=15)
    ax.axis('off')
    ax.legend(loc='lower left', framealpha=0.9)
    plt.tight_layout()
    plt.show()

plotar_topologia_rstp(G_rede, ativas, bloqueadas, root, "Topologia L2 SCADA-Core: Estado Nominal com Loops Bloqueados")

---
## 4. Simulação de Ruptura Física de Cabo (*Link Down*) e Reconvergência Rápida

Simulamos o rompimento do enlace troncal principal de $10\text{ Gbps}$ entre a Sala de Controle (`SW-CORE`) e o Reator R-200 (`SW-ZONA2`).
O protocolo RSTP detecta a queda do enlace e transita a porta alternativa da Zona 1 (`SW-ZONA1 <---> SW-ZONA2`) do estado *Discarding* para *Forwarding* em menos de $50\text{ ms}$.

In [ ]:
print("⚠️ SIMULAÇÃO: Ruptura física acidental do enlace SW-CORE <---> SW-ZONA2 (Tronco 10 Gbps)!")

G_falha = G_rede.copy()
G_falha.remove_edge("SW-CORE", "SW-ZONA2")

assert G_falha.number_of_edges() == G_rede.number_of_edges() - 1, "Enlace deve ser removido."
assert nx.is_connected(G_falha), "A rede redundante deve continuar conexa mesmo após a falha!"

# Executar reconvergência do RSTP
root_falha, custos_falha, ativas_falha, bloqueadas_falha, df_falha = executar_rstp(G_falha)

print("\n=================================================================")
print("RECONVERGÊNCIA RSTP PÓS-FALHA (COMUTAÇÃO RÁPIDA)")
print("=================================================================")
print(f"Root Bridge Mantido : {root_falha}")
print("Novos Custos Acumulados até o Root Bridge:")
for sw, c in custos_falha.items():
    print(f"  • {sw:10s}: Custo = {c}")

# Verificar que o enlace SW-ZONA1 <--> SW-ZONA2 foi desbloqueado
enlace_reaberto = ("SW-ZONA1", "SW-ZONA2") in ativas_falha or ("SW-ZONA2", "SW-ZONA1") in ativas_falha
assert enlace_reaberto, "O enlace alternativo entre SW-ZONA1 e SW-ZONA2 deve assumir Forwarding imediatamente!"
print("\n⚡ Enlace alternativo SW-ZONA1 <---> SW-ZONA2 desbloqueado para FORWARDING com sucesso!")

df_falha

---
## 5. Visualização Gráfica da Topologia Reconvergida Pós-Falha

Exibição do fluxo de dados reorganizado após a desconexão do cabo troncal.

In [ ]:
plotar_topologia_rstp(G_falha, ativas_falha, bloqueadas_falha, root_falha, "Topologia L2 SCADA-Core: Reconvergência RSTP após Ruptura do Tronco 10G")

---
## 6. Auditoria Formal em Matemática Discreta e Teoria dos Grafos

Comprovação rigorosa de que o grafo ativo gerado pelo RSTP cumpre as definições de uma **Árvore Geradora (*Spanning Tree*)** tanto no estado nominal quanto em contingência.

In [ ]:
def auditar_arvore_geradora(grafo_original: nx.Graph, arestas_ativas: List) -> Dict[str, Any]:
    T = nx.Graph()
    T.add_nodes_from(grafo_original.nodes())
    T.add_edges_from(arestas_ativas)
    
    v_count = T.number_of_nodes()
    e_count = T.number_of_edges()
    is_conn = nx.is_connected(T)
    is_tree = nx.is_tree(T)
    num_ciclos = len(nx.cycle_basis(T))
    
    return {
        "Vértices (|V|)": v_count,
        "Arestas Ativas (|E_T|)": e_count,
        "Critério |E_T| == |V| - 1": (e_count == v_count - 1),
        "É Conexo?": is_conn,
        "É Acíclico (Ciclos=0)?": (num_ciclos == 0),
        "É Árvore Geradora Formal?": is_tree
    }

audit_nominal = auditar_arvore_geradora(G_rede, ativas)
audit_falha = auditar_arvore_geradora(G_falha, ativas_falha)

df_auditoria = pd.DataFrame([audit_nominal, audit_falha], index=["Estado Nominal", "Pós-Ruptura de Link"])
print("\n=======================================================")
print("AUDITORIA MATEMÁTICA: PROPRIEDADES DA ÁRVORE GERADORA")
print("=======================================================")
print(df_auditoria.to_string())

# Validação estrita por asserts
assert audit_nominal["É Árvore Geradora Formal?"], "A topologia nominal deve ser formalmente uma árvore."
assert audit_falha["É Árvore Geradora Formal?"], "A topologia em falha deve ser formalmente uma árvore."
assert audit_nominal["Critério |E_T| == |V| - 1"] and audit_falha["Critério |E_T| == |V| - 1"]
print("\n✅ Todos os axiomas de Árvore Geradora foram validados matematicamente com SUCESSO!")

---
## 7. Conclusões e Diretrizes de Engenharia de Automação

1. **Eliminação de Tempestades de Broadcast:** A modelagem demonstrou que o bloqueio das portas redundantes (*Alternate Ports / Discarding*) impede a formação de loops na Camada 2, garantindo deterministicamente a disponibilidade do barramento SCADA-Core;
2. **Tolerância a Falhas em Tempo Real:** A simulação de rompimento do cabo troncal de $10\text{ Gbps}$ do reator R-200 comprovou a eficiência da reconvergência do RSTP, que reconfigura a árvore em milissegundos sem particionar a rede nem disparar desarmes indevidos nos CLPs;
3. **Separação de Camadas (LOPA):** Embora a rede Ethernet industrial proporcione disponibilidade superior a $99.99\%$, os laços críticos de emergência (botoeira `ESD-100` e desarme por sobrepressão `PT-201`) são mantidos em conexões físicas dedicadas (*hardwired*), em estrita conformidade com as normas **IEC 61508 / IEC 61511**.